In [3]:
#!/usr/bin/env python3
import os
import glob
import re
import numpy as np
import xarray as xr

RAW_Q_DIR = "/work/uc1275/u301827/02_MSE/full_midlatitude/raw/q"
OUT_DIR   = "/work/uc1275/u301827/02_MSE/full_midlatitude/TXX/random_q_null"
os.makedirs(OUT_DIR, exist_ok=True)

YEARS = range(1940, 2025)   # 1940–2025 inclusive
JJA_MONTHS = [6, 7, 8]

Q_NAME = "q"      # change if variable name differs
N_REALIZATIONS = 500       # number of Monte Carlo random datasets

def files_for_year(year):
    files = []
    for month in JJA_MONTHS:
        pattern = os.path.join(
            RAW_Q_DIR,
            f"q_{year}-{month:02d}_at_tasmax.nc"
        )
        matches = glob.glob(pattern)
        if len(matches) == 0:
            print(f"[WARN] missing {pattern}")
        else:
            files.extend(matches)
    return sorted(files)

def random_q_for_year(year, seed=None):
    """
    For one year:
    - open JJA monthly q_at_tasmax files
    - concatenate along time
    - randomly choose one JJA time index per grid cell
    - return q_random(lat, lon)
    """
    rng = np.random.default_rng(seed)

    files = files_for_year(year)
    if len(files) == 0:
        raise FileNotFoundError(f"No JJA q files found for {year}")

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        chunks={"time": -1}
    )

    if Q_NAME not in ds:
        raise KeyError(f"{Q_NAME} not found in {files[0]}. Available: {list(ds.data_vars)}")

    q = ds[Q_NAME]

    if "time" not in q.dims:
        raise ValueError(f"{Q_NAME} has no time dimension. Dims are {q.dims}")

    spatial_dims = [d for d in q.dims if d != "time"]

    ntime = q.sizes["time"]

    # Random time index per grid cell
    random_idx = xr.DataArray(
        rng.integers(0, ntime, size=tuple(q.sizes[d] for d in spatial_dims)),
        coords={d: q[d] for d in spatial_dims},
        dims=spatial_dims,
        name="random_jja_time_index"
    )

    q_random = q.isel(time=random_idx).astype("float32")
    q_random = q_random.rename("q_random_jja_at_tasmax")

    # Optional: store selected date
    selected_time = ds["time"].isel(time=random_idx)

    out = xr.Dataset()
    out["q_random_jja_at_tasmax"] = q_random
    out["random_jja_time_index"] = random_idx.astype("int16")

    if np.issubdtype(selected_time.dtype, np.datetime64):
        out["random_jja_yyyymmdd"] = selected_time.dt.strftime("%Y%m%d").astype("int32")
    else:
        out["random_jja_time_str"] = selected_time.astype(str)

    out = out.expand_dims(year=[year])

    ds.close()
    return out

def create_one_realization(realization):
    #print(f"[REALIZATION] {realization}")

    yearly = []

    for year in YEARS:
        try:
            ds_year = random_q_for_year(
                year,
                seed=realization * 100000 + year
            )
            yearly.append(ds_year)
            #print(f"  OK {year}")
        except Exception as e:
            print(f"  SKIP {year}: {e}")

    if not yearly:
        raise RuntimeError("No yearly datasets created.")

    ds_out = xr.concat(yearly, dim="year").sortby("year")

    ds_out.attrs.update({
        "description": (
            "Monte Carlo null dataset: for each year and grid cell, "
            "q_at_tasmax is randomly sampled from all JJA days of the same year."
        ),
        "sampling": "random JJA q_at_tasmax per year and grid cell",
        "realization": realization,
    })

    out_file = os.path.join(
        OUT_DIR,
        f"q_random_jja_at_tasmax_realization_{realization:04d}.nc"
    )

    encoding = {
        v: {"zlib": True, "complevel": 4}
        for v in ds_out.data_vars
    }

    ds_out.to_netcdf(out_file, encoding=encoding)
    ds_out.close()

    #print(f"[SAVED] {out_file}")

from joblib import Parallel, delayed
import multiprocessing

# -----------------------
# Continue from existing realizations
# -----------------------
realization_re = re.compile(
    r"q_random_jja_at_tasmax_realization_(\d{4})\.nc$"
)

existing_files = glob.glob(
    os.path.join(OUT_DIR, "q_random_jja_at_tasmax_realization_*.nc")
)

existing_realizations = []

for f in existing_files:
    m = realization_re.search(os.path.basename(f))
    if m:
        existing_realizations.append(int(m.group(1)))

if existing_realizations:
    start_realization = max(existing_realizations) + 1
else:
    start_realization = 0

end_realization = start_realization + N_REALIZATIONS

print(f"[INFO] Found {len(existing_realizations)} existing realizations")
print(f"[INFO] Starting at realization {start_realization}")
print(f"[INFO] Creating realizations {start_realization} to {end_realization - 1}")

# choose number of workers
n_jobs = 20

Parallel(n_jobs=n_jobs, backend="loky")(
    delayed(create_one_realization)(realization)
    for realization in range(start_realization, end_realization)
)

[INFO] Found 500 existing realizations
[INFO] Starting at realization 500
[INFO] Creating realizations 500 to 999
[REALIZATION] 496


/home/u/u301827/.conda/envs/thesis/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[REALIZATION] 498
[REALIZATION] 492
[REALIZATION] 482
[REALIZATION] 487
[REALIZATION] 494
[REALIZATION] 499
[REALIZATION] 481
[REALIZATION] 493
[REALIZATION] 497
[REALIZATION] 488
[REALIZATION] 483
[REALIZATION] 480
[REALIZATION] 484
[REALIZATION] 495
[REALIZATION] 490
[REALIZATION] 485
[REALIZATION] 489
[REALIZATION] 491
[REALIZATION] 486


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [1]:
#!/usr/bin/env python3
import os
import glob
import re
import numpy as np
import xarray as xr

RAW_Q_DIR = "/work/uc1275/u301827/02_MSE/full_midlatitude/raw/t"
OUT_DIR   = "/work/uc1275/u301827/02_MSE/full_midlatitude/TXX/random_t_null"
os.makedirs(OUT_DIR, exist_ok=True)

YEARS = range(1940, 2025)   # 1940–2025 inclusive
JJA_MONTHS = [6, 7, 8]

Q_NAME = "t"      # change if variable name differs
N_REALIZATIONS = 1000       # number of Monte Carlo random datasets

def files_for_year(year):
    files = []
    for month in JJA_MONTHS:
        pattern = os.path.join(
            RAW_Q_DIR,
            f"t_{year}-{month:02d}_dailymax.nc"
        )
        matches = glob.glob(pattern)
        if len(matches) == 0:
            print(f"[WARN] missing {pattern}")
        else:
            files.extend(matches)
    return sorted(files)

def random_q_for_year(year, seed=None):
    """
    For one year:
    - open JJA monthly q_at_tasmax files
    - concatenate along time
    - randomly choose one JJA time index per grid cell
    - return qt_random(lat, lon)
    """
    rng = np.random.default_rng(seed)

    files = files_for_year(year)
    if len(files) == 0:
        raise FileNotFoundError(f"No JJA q files found for {year}")

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        chunks={"time": -1}
    )

    if Q_NAME not in ds:
        raise KeyError(f"{Q_NAME} not found in {files[0]}. Available: {list(ds.data_vars)}")

    q = ds[Q_NAME]

    if "time" not in q.dims:
        raise ValueError(f"{Q_NAME} has no time dimension. Dims are {q.dims}")

    spatial_dims = [d for d in q.dims if d != "time"]

    ntime = q.sizes["time"]

    # Random time index per grid cell
    random_idx = xr.DataArray(
        rng.integers(0, ntime, size=tuple(q.sizes[d] for d in spatial_dims)),
        coords={d: q[d] for d in spatial_dims},
        dims=spatial_dims,
        name="random_jja_time_index"
    )

    q_random = q.isel(time=random_idx).astype("float32")
    q_random = q_random.rename("q_random_jja_at_tasmax")

    # Optional: store selected date
    selected_time = ds["time"].isel(time=random_idx)

    out = xr.Dataset()
    out["t_random_jja_dailymax"] = q_random
    out["random_jja_time_index"] = random_idx.astype("int16")

    if np.issubdtype(selected_time.dtype, np.datetime64):
        out["random_jja_yyyymmdd"] = selected_time.dt.strftime("%Y%m%d").astype("int32")
    else:
        out["random_jja_time_str"] = selected_time.astype(str)

    out = out.expand_dims(year=[year])

    ds.close()
    return out

def create_one_realization(realization):
    #print(f"[REALIZATION] {realization}")

    yearly = []

    for year in YEARS:
        try:
            ds_year = random_q_for_year(
                year,
                seed=realization * 100000 + year
            )
            yearly.append(ds_year)
            #print(f"  OK {year}")
        except Exception as e:
            print(f"  SKIP {year}: {e}")

    if not yearly:
        raise RuntimeError("No yearly datasets created.")

    ds_out = xr.concat(yearly, dim="year").sortby("year")

    ds_out.attrs.update({
        "description": (
            "Monte Carlo null dataset: for each year and grid cell, "
            "q_at_tasmax is randomly sampled from all JJA days of the same year."
        ),
        "sampling": "random JJA q_at_tasmax per year and grid cell",
        "realization": realization,
    })

    out_file = os.path.join(
        OUT_DIR,
        f"q_random_jja_at_tasmax_realization_{realization:04d}.nc"
    )

    encoding = {
        v: {"zlib": True, "complevel": 4}
        for v in ds_out.data_vars
    }

    ds_out.to_netcdf(out_file, encoding=encoding)
    ds_out.close()

    #print(f"[SAVED] {out_file}")

from joblib import Parallel, delayed
import multiprocessing

# -----------------------
# Continue from existing realizations
# -----------------------
realization_re = re.compile(
    r"q_random_jja_at_tasmax_realization_(\d{4})\.nc$"
)

existing_files = glob.glob(
    os.path.join(OUT_DIR, "q_random_jja_at_tasmax_realization_*.nc")
)

existing_realizations = []

for f in existing_files:
    m = realization_re.search(os.path.basename(f))
    if m:
        existing_realizations.append(int(m.group(1)))

if existing_realizations:
    start_realization = max(existing_realizations) + 1
else:
    start_realization = 0

end_realization = start_realization + N_REALIZATIONS

print(f"[INFO] Found {len(existing_realizations)} existing realizations")
print(f"[INFO] Starting at realization {start_realization}")
print(f"[INFO] Creating realizations {start_realization} to {end_realization - 1}")

# choose number of workers
n_jobs = 20

Parallel(n_jobs=n_jobs, backend="loky")(
    delayed(create_one_realization)(realization)
    for realization in range(start_realization, end_realization)
)

[INFO] Found 0 existing realizations
[INFO] Starting at realization 0
[INFO] Creating realizations 0 to 999


/home/u/u301827/.conda/envs/thesis/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,